# Phase 6: 構造化RAG 完全評価 (v6.2.1)

**修正内容**:
- コンテキスト構築ロジックの改善
- ベクトル検索を常に補完として追加（フォールバックではなく）
- Phase 6.1の改善を維持しつつ、Phase 6.2の新機能を統合

**作成日**: 2026-01-23

## Section 1: 環境セットアップ

In [ ]:
# 1.1 パッケージインストール
%%capture
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma
!pip install -q chromadb sentence-transformers tqdm pandas
print("パッケージインストール完了")

In [ ]:
# 1.2 GPU確認・メモリ管理
import torch, gc

def print_memory():
    if torch.cuda.is_available(): print(f"GPU VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")
    import psutil; print(f"RAM: {psutil.virtual_memory().percent}%")

def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")
print_memory()

In [ ]:
# 1.3 Google Driveマウント
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = "/content/drive/MyDrive/experiments-local-llm"
DATA_DIR, RESULTS_DIR = f"{BASE_DIR}/data", f"{BASE_DIR}/results"
for d in [DATA_DIR, RESULTS_DIR]: os.makedirs(d, exist_ok=True)
sys.path.insert(0, BASE_DIR)
print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
# 1.4 モデル設定
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
print(f"LLM: {LLM_MODEL}\nEmbedding: {EMBEDDING_MODEL}")

## Section 2: データ読み込み

In [ ]:
# 2.1 POIデータ読み込み
import json
from collections import Counter

with open(f"{DATA_DIR}/poi_documents.json", "r", encoding="utf-8") as f:
    poi_documents = json.load(f)

all_pois = []
for doc in poi_documents:
    poi = doc["metadata"].copy() if "metadata" in doc else doc.copy()
    poi["content"] = doc.get("content", "")
    all_pois.append(poi)
print(f"POIデータ: {len(all_pois)}件")

In [ ]:
# 2.2 Phase 6モジュール読み込み
from src.geo_utils import (enrich_all_pois, get_nearest_pois, filter_by_radius,
    compare_by_radius, generate_proximity_context, generate_sensitivity_context)
from src.aggregator import (compare_east_west, get_top_categories,
    analyze_category_by_direction, filter_by_category)
from src.structured_rag_system import analyze_question

enriched_pois = enrich_all_pois(all_pois)
print(f"空間情報追加完了: {len(enriched_pois)}件")

# Phase 6.2機能テスト
nearest = get_nearest_pois(enriched_pois, category="カフェ", top_n=3)
print(f"\n最寄りカフェTOP3:")
for i, c in enumerate(nearest, 1): print(f"  {i}. {c['name']} - {c['distance_from_station']:.0f}m")

comp = compare_by_radius(enriched_pois, 300, 500, "カフェ")
print(f"\n半径比較: {comp.to_japanese()}")

In [ ]:
# 2.3 テストケース読み込み
try:
    from src.test_cases_v2 import TEST_CASES_V2
    print(f"テストケース: {len(TEST_CASES_V2)}件")
    
    # レベル別・サブカテゴリ別件数
    level_counts = Counter()
    subcat_counts = Counter()
    for tc in TEST_CASES_V2:
        level = getattr(tc, 'level', None) or getattr(tc, 'difficulty', None) or 'unknown'
        subcat = getattr(tc, 'subcategory', None) or getattr(tc, 'sub_category', None) or 'unknown'
        level_counts[level] += 1
        subcat_counts[subcat] += 1
    
    print("\nレベル別:")
    for level in sorted(level_counts.keys()): print(f"  {level}: {level_counts[level]}件")
    
    # ヘルパー関数
    def get_test_cases_by_level(level):
        return [tc for tc in TEST_CASES_V2 
                if getattr(tc, 'level', None) == level or getattr(tc, 'difficulty', None) == level]
    
    def get_test_cases_by_subcategory(subcat):
        return [tc for tc in TEST_CASES_V2 
                if getattr(tc, 'subcategory', None) == subcat or getattr(tc, 'sub_category', None) == subcat]

except ImportError as e:
    print(f"エラー: {e}")
    TEST_CASES_V2 = None

## Section 3: モデルセットアップ

In [ ]:
# 3.1 Embeddingモデル
from langchain_huggingface import HuggingFaceEmbeddings
print("Embeddingモデルロード中...")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL,
    model_kwargs={'device': DEVICE}, encode_kwargs={'normalize_embeddings': True})
print("完了"); print_memory()

In [ ]:
# 3.2 ベクトルストア構築（メタデータフィルタリング対応）
from langchain_chroma import Chroma
from langchain_core.documents import Document

def flatten_metadata(metadata):
    """ネストした辞書やリストをフラット化"""
    flat = {}
    for key, value in metadata.items():
        if isinstance(value, dict):
            flat[key] = json.dumps(value, ensure_ascii=False)
        elif isinstance(value, list):
            flat[key] = json.dumps(value, ensure_ascii=False)
        elif value is None:
            flat[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            flat[key] = value
        else:
            flat[key] = str(value)
    return flat

print("ベクトルストア構築中...")
documents = []
for poi in poi_documents:
    if "metadata" in poi:
        content = poi.get("content", f"{poi['metadata'].get('name', '')}")
        metadata = flatten_metadata(poi["metadata"])
    else:
        content = poi.get("content", f"{poi.get('name', '')}")
        metadata = flatten_metadata(poi)
    if "content" in metadata: del metadata["content"]
    documents.append(Document(page_content=content, metadata=metadata))

vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings, collection_name="poi_phase6")
print(f"完了: {len(documents)}件"); print_memory()

In [ ]:
# 3.3 Embedding解放
del embeddings; clear_memory(); print_memory()

In [ ]:
# 3.4 LLMモデルロード
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
print(f"LLMロード中: {LLM_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, quantization_config=quantization_config,
    device_map="auto", trust_remote_code=True, low_cpu_mem_usage=True)
print("完了"); print_memory()

## Section 4: RAGシステム (Phase 6.2.1 修正版)

In [ ]:
# 4.1 構造化RAGシステム（Phase 6.2.1修正版）
import time

class StructuredRAGEvaluator:
    """構造化RAG評価用システム（Phase 6.2.1修正版）
    
    修正点:
    - ベクトル検索を常に補完として追加（フォールバックではなく）
    - 構造化コンテキストとベクトル検索結果を統合
    """
    
    def __init__(self, model, tokenizer, vectorstore, all_pois):
        self.model = model
        self.tokenizer = tokenizer
        self.vectorstore = vectorstore
        self.all_pois = all_pois
        self.system_prompt = """あなたは渋谷エリアの地理情報に詳しいアシスタントです。
提供された情報に基づいて、正確かつ簡潔に回答してください。
座標情報がある場合は必ず含めてください。
数値データがある場合は具体的な数字を使って回答してください。"""
    
    def _generate(self, prompt, max_tokens=512):
        messages = [{"role":"system","content":self.system_prompt},{"role":"user","content":prompt}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.1,
                do_sample=True, pad_token_id=self.tokenizer.eos_token_id)
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response.split("assistant")[-1].strip() if "assistant" in response.lower() else response
    
    def _get_vector_search_context(self, question, k=5):
        """ベクトル検索結果をコンテキスト形式で取得"""
        try:
            results = self.vectorstore.similarity_search(question, k=k)
            if not results:
                return ""
            
            lines = ["【関連POI情報】"]
            for r in results:
                name = r.metadata.get('name', '不明')
                category = r.metadata.get('category', '')
                lat = r.metadata.get('lat', '')
                lon = r.metadata.get('lon', '')
                
                line = f"- {name}"
                if category:
                    line += f" ({category})"
                if lat and lon:
                    line += f" 座標: ({lat}, {lon})"
                lines.append(line)
            
            return "\n".join(lines)
        except Exception as e:
            print(f"ベクトル検索エラー: {e}")
            return ""
    
    def _build_context(self, question, analysis):
        """質問分析に基づいてコンテキストを構築（Phase 6.2.1修正版）
        
        修正点:
        - 構造化コンテキストを構築
        - ベクトル検索結果を常に追加（補完として）
        """
        structured_parts = []  # 構造化データからのコンテキスト
        
        # カテゴリ特定
        cat = analysis.subcategories[0] if analysis.subcategories else None
        
        # === Phase 6.2: 近接性検索 ===
        if analysis.requires_proximity and cat:
            structured_parts.append(generate_proximity_context(self.all_pois, cat, top_n=5))
        
        # === Phase 6.2: 感度分析 ===
        if analysis.requires_sensitivity and cat:
            r1, r2 = analysis.sensitivity_radii if analysis.sensitivity_radii else (300, 500)
            structured_parts.append(generate_sensitivity_context(self.all_pois, cat, r1, r2))
            comp = compare_by_radius(self.all_pois, r1, r2, cat)
            if comp.count1 > 0:
                conclusion = "半径を変えると件数が大きく変化するため、結論は条件に依存します。" if comp.ratio >= 1.5 else "半径を変えても結論は大きく変わりません。"
                structured_parts.append(f"\n【結論】\n{conclusion}")
        
        # === Phase 6.1: 東西比較 ===
        if analysis.requires_comparison and "東" in question and "西" in question:
            result = compare_east_west(self.all_pois, cat)
            structured_parts.append(f"【東西比較】\n{result.to_japanese()}")
            if cat:
                detail = analyze_category_by_direction(self.all_pois, cat)
                structured_parts.append("\n方向別詳細:")
                for d, c in detail['by_direction'].items():
                    structured_parts.append(f"  {d}: {c}件")
        
        # === Phase 6.1: 集計 ===
        if analysis.requires_aggregation:
            if cat:
                filtered = filter_by_category(self.all_pois, cat)
                structured_parts.append(f"【{cat}の集計】\n総数: {len(filtered)}件")
                # 例も追加
                if filtered:
                    examples = [p.get('name', '不明') for p in filtered[:5]]
                    structured_parts.append(f"例: {', '.join(examples)}")
            else:
                top = get_top_categories(self.all_pois, 5)
                structured_parts.append("【カテゴリランキング】")
                for i, c in enumerate(top, 1):
                    structured_parts.append(f"  {i}. {c.category}: {c.count}件")
        
        # === ベクトル検索（常に追加 - 補完として） ===
        vector_context = self._get_vector_search_context(question, k=5)
        
        # コンテキスト統合
        all_parts = []
        
        # 構造化コンテキストがあれば先に追加
        if structured_parts:
            all_parts.extend(structured_parts)
        
        # ベクトル検索結果を常に追加
        if vector_context:
            if structured_parts:
                all_parts.append("")  # 空行で区切り
            all_parts.append(vector_context)
        
        return "\n".join(all_parts)
    
    def query(self, question):
        """構造化RAGで質問に回答"""
        start = time.time()
        
        analysis = analyze_question(question)
        context = self._build_context(question, analysis)
        
        prompt = f"""以下の情報を参考にして質問に回答してください。

{context}

【質問】
{question}

【回答】
上記の情報を基に、具体的な数値や座標を含めて回答します。"""
        
        answer = self._generate(prompt)
        elapsed = time.time() - start
        
        return {
            "answer": answer,
            "analysis": analysis.to_dict(),
            "context": context,
            "time_sec": round(elapsed, 2)
        }

# システム初期化
rag_system = StructuredRAGEvaluator(model, tokenizer, vectorstore, enriched_pois)
print("RAGシステム初期化完了（Phase 6.2.1修正版）")

In [ ]:
# 4.2 動作確認（修正確認用）
print("=== Phase 6.2.1 動作確認 ===")

# 基本検索（basic_location）- Phase 6.2で悪化した部分の確認
q0 = "渋谷駅の場所を教えてください"
r0 = rag_system.query(q0)
print(f"\n[基本検索] Q: {q0}")
print(f"Type: {r0['analysis']['question_type']}")
print(f"Context preview: {r0['context'][:200]}...")
print(f"A: {r0['answer'][:200]}...")

# 近接性検索（Phase 6.2の改善を維持）
q1 = "渋谷駅に最も近いコンビニはどこですか？"
r1 = rag_system.query(q1)
print(f"\n[近接性] Q: {q1}")
print(f"Type: {r1['analysis']['question_type']}")
print(f"A: {r1['answer'][:200]}...")

# 感度分析（Phase 6.2の改善を維持）
q2 = "渋谷駅周辺はカフェが多いという結論は、半径を500mから300mに変えても成立しますか？"
r2 = rag_system.query(q2)
print(f"\n[感度分析] Q: {q2[:50]}...")
print(f"Type: {r2['analysis']['question_type']}")
print(f"A: {r2['answer'][:200]}...")

# 東西比較（Phase 6.1の改善を維持）
q3 = "渋谷駅の東側と西側、どちらにカフェが多いですか？"
r3 = rag_system.query(q3)
print(f"\n[東西比較] Q: {q3}")
print(f"Type: {r3['analysis']['question_type']}")
print(f"A: {r3['answer'][:200]}...")

## Section 5: 評価実行

In [ ]:
# 5.1 評価関数
import re
from tqdm import tqdm

def evaluate_response(response, tc):
    answer = response.get("answer", "")
    kws = getattr(tc, 'expected_keywords', []) or []
    kw_score = sum(1 for kw in kws if kw in answer) / len(kws) * 100 if kws else 50
    coord_score = 100 if "35." in answer and "139." in answer else 0
    num_score = 100 if re.findall(r'\d+', answer) else 0
    total = kw_score * 0.4 + coord_score * 0.3 + num_score * 0.3
    return {"keyword_score": kw_score, "coord_score": coord_score, "number_score": num_score, "total_score": round(total, 1)}

def run_evaluation(rag, cases, max_cases=None):
    results = []
    for tc in tqdm(cases[:max_cases] if max_cases else cases, desc="評価中"):
        try:
            resp = rag.query(tc.prompt)
            ev = evaluate_response(resp, tc)
            level = getattr(tc, 'level', None) or getattr(tc, 'difficulty', None) or 'unknown'
            category = getattr(tc, 'category', None) or 'unknown'
            subcategory = getattr(tc, 'subcategory', None) or getattr(tc, 'sub_category', None) or 'unknown'
            results.append({"id": tc.id, "level": level, "category": category, "subcategory": subcategory,
                "prompt": tc.prompt, "answer": resp["answer"][:500], "time_sec": resp.get("time_sec",0),
                "analysis": resp.get("analysis",{}), "scores": ev})
            clear_memory()
        except Exception as e:
            print(f"Error ({tc.id}): {e}")
            results.append({"id": tc.id, "level": "unknown", "error": str(e)})
    return results

In [ ]:
# 5.2 サブカテゴリ別テスト（修正確認用）
if TEST_CASES_V2:
    print("=== Phase 6.2.1 サブカテゴリ別テスト ===")
    
    # 悪化したサブカテゴリと改善したサブカテゴリの両方をテスト
    test_subcats = [
        "basic_location",      # P6.2で-34pt悪化
        "basic_category",      # P6.2で-30pt悪化
        "spatial_proximity",   # P6.2で+13.6pt改善
        "advanced_sensitivity", # P6.2で+18.6pt改善
        "spatial_comparison",  # P6.1で大幅改善
    ]
    
    subcat_results = {}
    for subcat in test_subcats:
        cases = get_test_cases_by_subcategory(subcat)
        print(f"\n{subcat}: {len(cases)}件")
        if cases:
            results = run_evaluation(rag_system, cases)
            scores = [r['scores']['total_score'] for r in results if 'error' not in r]
            avg = round(sum(scores)/len(scores), 1) if scores else 0
            subcat_results[subcat] = avg
            print(f"  平均: {avg}pt")
            for r in results:
                if "error" not in r:
                    print(f"    {r['id']}: {r['scores']['total_score']:.1f}pt")
    
    # Phase 6.2との比較
    phase62_scores = {
        "basic_location": 62.8,
        "basic_category": 51.3,
        "spatial_proximity": 68.4,
        "advanced_sensitivity": 67.3,
        "spatial_comparison": 64.0,
    }
    
    print("\n--- Phase 6.2 vs 6.2.1 比較 ---")
    print(f"{'サブカテゴリ':<25} {'P6.2':>8} {'P6.2.1':>8} {'Δ':>8}")
    print("-" * 50)
    for subcat in test_subcats:
        p62 = phase62_scores.get(subcat, 0)
        p621 = subcat_results.get(subcat, 0)
        diff = p621 - p62
        marker = "✅" if diff > 0 else "❌" if diff < 0 else "-"
        print(f"{subcat:<25} {p62:>8.1f} {p621:>8.1f} {diff:>+8.1f} {marker}")

In [ ]:
# 5.3 全テスト実行
if TEST_CASES_V2:
    print(f"全テスト実行: {len(TEST_CASES_V2)}件（約30-40分）")
    all_results_rag = run_evaluation(rag_system, TEST_CASES_V2)
    print(f"完了: {len(all_results_rag)}件")

## Section 6: 結果分析

In [ ]:
# 6.1 分析
def analyze_results(results):
    valid = [r for r in results if "error" not in r]
    level_scores, subcat_scores = {}, {}
    levels = sorted(set(r["level"] for r in valid))
    for level in levels:
        lr = [r for r in valid if r["level"]==level]
        if lr: level_scores[level] = {"count":len(lr), "avg":round(sum(r["scores"]["total_score"] for r in lr)/len(lr),1)}
    for r in valid:
        sc = r["subcategory"]
        subcat_scores.setdefault(sc, []).append(r["scores"]["total_score"])
    subcat_avg = {sc: round(sum(s)/len(s),1) for sc, s in subcat_scores.items()}
    all_scores = [r["scores"]["total_score"] for r in valid]
    times = [r["time_sec"] for r in valid if "time_sec" in r]
    return {"overall": {"count":len(all_scores), "avg":round(sum(all_scores)/len(all_scores),1) if all_scores else 0},
            "by_level": level_scores, "by_subcategory": subcat_avg, "avg_time_sec": round(sum(times)/len(times),1) if times else 0}

if 'all_results_rag' in dir():
    analysis = analyze_results(all_results_rag)
    print(f"=== Phase 6.2.1 結果 ===")
    print(f"全体: {analysis['overall']['avg']}pt / {analysis['avg_time_sec']}秒")
    print("\nレベル別:"); [print(f"  {l}: {s['avg']}pt") for l,s in analysis['by_level'].items()]
    print("\nサブカテゴリ別:"); [print(f"  {sc}: {avg}pt") for sc,avg in sorted(analysis['by_subcategory'].items(), key=lambda x:x[1], reverse=True)]

In [ ]:
# 6.2 Phase比較（P5 vs P6.1 vs P6.2 vs P6.2.1）
phase5 = {"basic_location":71.7,"basic_category":58.3,"spatial_proximity":61.7,"spatial_density":65.0,
    "spatial_comparison":51.4,"constraint_single":53.3,"constraint_multi":53.3,"decision_location":63.3,
    "decision_business":63.3,"advanced_sensitivity":60.0,"advanced_comparison":59.5,"advanced_uncertainty":66.7}
phase61 = {"basic_location":96.8,"basic_category":81.3,"spatial_proximity":54.8,"spatial_density":62.8,
    "spatial_comparison":76.0,"constraint_single":51.3,"constraint_multi":80.0,"decision_location":57.6,
    "decision_business":83.5,"advanced_sensitivity":48.7,"advanced_comparison":64.4,"advanced_uncertainty":67.3}
phase62 = {"basic_location":62.8,"basic_category":51.3,"spatial_proximity":68.4,"spatial_density":66.4,
    "spatial_comparison":64.0,"constraint_single":70.0,"constraint_multi":70.0,"decision_location":59.2,
    "decision_business":65.5,"advanced_sensitivity":67.3,"advanced_comparison":68.3,"advanced_uncertainty":54.7}

if 'analysis' in dir():
    print(f"\n{'サブカテゴリ':<25} {'P5':>6} {'P6.1':>6} {'P6.2':>6} {'P6.2.1':>6} {'Δ6.2.1':>7}")
    print("-"*65)
    for sc in phase5:
        p5, p61, p62 = phase5[sc], phase61.get(sc,0), phase62.get(sc,0)
        p621 = analysis['by_subcategory'].get(sc, 0)
        diff = p621 - p62
        m = "✅" if diff > 5 else "❌" if diff < -5 else "→"
        print(f"{sc:<25} {p5:>6.1f} {p61:>6.1f} {p62:>6.1f} {p621:>6.1f} {diff:>+7.1f} {m}")
    
    # 全体比較
    print(f"\n--- 全体スコア推移 ---")
    print(f"Phase 5:   60.3pt")
    print(f"Phase 6.1: 69.6pt (+9.3)")
    print(f"Phase 6.2: 64.1pt (-5.5)")
    print(f"Phase 6.2.1: {analysis['overall']['avg']}pt ({analysis['overall']['avg']-64.1:+.1f})")

## Section 7: 結果保存

In [ ]:
# 7.1 JSON保存
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

if 'all_results_rag' in dir() and 'analysis' in dir():
    data = {"timestamp":timestamp, "phase":"6.2.1", "model":LLM_MODEL, "poi_count":len(enriched_pois),
            "analysis":analysis, "phase5":phase5, "phase61":phase61, "phase62":phase62, "results":all_results_rag}
    path = f"{RESULTS_DIR}/phase621_eval_{timestamp}.json"
    with open(path, "w", encoding="utf-8") as f: json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"保存: {path}")

In [ ]:
# 7.2 レポート生成
if 'analysis' in dir():
    # 改善/悪化サブカテゴリ
    improvements = []
    regressions = []
    for sc in phase62:
        p62, p621 = phase62[sc], analysis['by_subcategory'].get(sc, 0)
        diff = p621 - p62
        if diff > 5: improvements.append((sc, diff))
        elif diff < -5: regressions.append((sc, diff))
    
    report = f"""# Phase 6.2.1 評価レポート
**日時**: {timestamp} | **モデル**: {LLM_MODEL} | **POI**: {len(enriched_pois)}件

## 全体スコア推移
| Phase | スコア | 変化 |
|-------|--------|------|
| Phase 5 | 60.3pt | - |
| Phase 6.1 | 69.6pt | +9.3pt |
| Phase 6.2 | 64.1pt | -5.5pt |
| **Phase 6.2.1** | **{analysis['overall']['avg']}pt** | **{analysis['overall']['avg']-64.1:+.1f}pt** |

## 平均処理時間: {analysis['avg_time_sec']}秒

## Phase 6.2 → 6.2.1 変化

### 改善 (+5pt以上)
"""
    for sc, diff in sorted(improvements, key=lambda x: x[1], reverse=True):
        report += f"- {sc}: {phase62[sc]}pt → {analysis['by_subcategory'].get(sc,0)}pt ({diff:+.1f}pt)\n"
    
    report += "\n### 悪化 (-5pt以上)\n"
    for sc, diff in sorted(regressions, key=lambda x: x[1]):
        report += f"- {sc}: {phase62[sc]}pt → {analysis['by_subcategory'].get(sc,0)}pt ({diff:+.1f}pt)\n"
    
    report += f"\n### 維持 (±5pt以内)\nその他のサブカテゴリは概ね維持\n"
    
    rpath = f"{RESULTS_DIR}/phase621_report_{timestamp}.md"
    with open(rpath, "w", encoding="utf-8") as f: f.write(report)
    print(f"レポート: {rpath}\n\n{report}")